# NB10A — Topic-Sentiment Keyword Taxonomy
## Topic-Sentiment Extension | MSc Dissertation
### Pavlos Papachristos | University of Essex

**Purpose:** Filter BIS speeches by topic using keyword taxonomy (Credit, Real Estate, 
Monetary Policy, Sovereign, Corporate). Produce topic-tagged speech subsets for 
FinBERT re-inference in NB10B.

In [2]:
import pandas as pd
import numpy as np
import os

# Paths
DATA_DIR = r"C:\Users\Owner\OneDrive\dissertation\data\processed"
RAW_DIR  = r"C:\Users\Owner\OneDrive\dissertation\data\raw\Bis_Org_Speaches"

# Load the BIS sentiment file produced by NB03
bis = pd.read_csv(os.path.join(DATA_DIR, "bis_sentiment_raw.csv"))
print(bis.shape)
print(bis.columns.tolist())
print(bis.head(3))B

(16622, 8)
['url', 'year', 'date', 'author', 'description', 'P_pos', 'P_neg', 'P_neutral']
                                       url  year                 date  \
0  https://www.bis.org/review/r970512a.pdf  1997  1997-04-24 00:00:00   
1  https://www.bis.org/review/r970605b.pdf  1997  1997-05-26 00:00:00   
2  https://www.bis.org/review/r971104a.pdf  1997  1997-10-28 00:00:00   

               author                                        description  \
0    Laurence H Meyer  Remarks by Mr. Laurence H. Meyer, a member of ...   
1     Lars Heikensten  Address by the Deputy Governor of the Bank of ...   
2  Graeme J. Thompson  Speech by the Deputy Governor of the Reserve B...   

      P_pos     P_neg  P_neutral  
0  0.132078  0.230044   0.637878  
1  0.067387  0.164713   0.767901  
2       NaN       NaN        NaN  


In [3]:
# ── Keyword Taxonomy ──────────────────────────────────────────────────────────
TOPICS = {
    "credit": [
        "credit", "lending", "loan", "bank credit", "credit growth",
        "credit cycle", "private credit", "credit expansion", "credit contraction",
        "credit boom", "credit bust", "debt", "leverage", "borrowing"
    ],
    "real_estate": [
        "house price", "housing", "real estate", "property price",
        "mortgage", "residential", "commercial property", "land price",
        "property market", "housing boom", "housing bubble"
    ],
    "monetary_policy": [
        "interest rate", "monetary policy", "central bank", "inflation",
        "price stability", "policy rate", "quantitative easing", "QE",
        "tightening", "easing", "forward guidance", "yield curve"
    ],
    "sovereign": [
        "sovereign", "government debt", "fiscal", "public debt",
        "sovereign risk", "sovereign spread", "government bond",
        "budget deficit", "debt sustainability", "sovereign default"
    ],
    "corporate": [
        "corporate", "firm", "business", "corporate debt", "corporate bond",
        "corporate sector", "non-financial", "enterprise", "default rate",
        "corporate leverage", "corporate credit"
    ]
}

def tag_topics(description, topics):
    """Return list of topics whose keywords appear in the description."""
    if not isinstance(description, str):
        return []
    desc_lower = description.lower()
    matched = [topic for topic, kws in topics.items()
               if any(kw in desc_lower for kw in kws)]
    return matched

bis["topics"] = bis["description"].apply(lambda x: tag_topics(x, TOPICS))
bis["n_topics"] = bis["topics"].apply(len)

# Summary
print("Speeches with at least one topic tag:", (bis["n_topics"] > 0).sum())
print("\nPer-topic counts:")
for t in TOPICS:
    n = bis["topics"].apply(lambda x: t in x).sum()
    print(f"  {t:20s}: {n}")

Speeches with at least one topic tag: 6957

Per-topic counts:
  credit              : 280
  real_estate         : 210
  monetary_policy     : 5654
  sovereign           : 69
  corporate           : 1128


In [4]:
# ── Expanded Keyword Taxonomy ─────────────────────────────────────────────────
TOPICS = {
    "credit": [
        "credit", "lending", "loan", "bank credit", "credit growth",
        "credit cycle", "private credit", "credit expansion", "credit contraction",
        "credit boom", "credit bust", "debt", "leverage", "borrowing",
        "financial stability", "bank lending", "credit risk", "systemic risk",
        "financial intermediation", "credit market", "credit conditions",
        "bank financing", "debt growth", "indebtedness", "overleveraged",
        "deleveraging", "credit crunch", "tightening credit", "credit supply"
    ],
    "real_estate": [
        "house price", "housing", "real estate", "property price",
        "mortgage", "residential", "commercial property", "land price",
        "property market", "housing boom", "housing bubble",
        "home price", "housing market", "property boom", "property bubble",
        "housing cycle", "real estate market", "housing finance",
        "housing wealth", "property valuation", "construction sector"
    ],
    "monetary_policy": [
        "interest rate", "monetary policy", "central bank", "inflation",
        "price stability", "policy rate", "quantitative easing", "QE",
        "tightening", "easing", "forward guidance", "yield curve",
        "monetary stance", "liquidity", "money supply", "repo rate",
        "bank rate", "overnight rate", "monetary transmission",
        "inflation target", "deflation", "disinflation", "monetary conditions"
    ],
    "sovereign": [
        "sovereign", "government debt", "fiscal", "public debt",
        "sovereign risk", "sovereign spread", "government bond",
        "budget deficit", "debt sustainability", "sovereign default",
        "public finance", "fiscal policy", "government borrowing",
        "debt-to-gdp", "fiscal consolidation", "fiscal stimulus",
        "government deficit", "treasury", "public sector debt",
        "sovereign bond", "fiscal sustainability", "debt burden"
    ],
    "corporate": [
        "corporate", "firm", "business", "corporate debt", "corporate bond",
        "corporate sector", "non-financial", "enterprise", "default rate",
        "corporate leverage", "corporate credit",
        "business investment", "corporate financing", "company debt",
        "corporate balance sheet", "business cycle", "corporate default",
        "private sector", "corporate funding", "business lending",
        "corporate borrowing", "company financing"
    ]
}

bis["topics"] = bis["description"].apply(lambda x: tag_topics(x, TOPICS))
bis["n_topics"] = bis["topics"].apply(len)

print("Speeches with at least one topic tag:", (bis["n_topics"] > 0).sum())
print("\nPer-topic counts:")
for t in TOPICS:
    n = bis["topics"].apply(lambda x: t in x).sum()
    print(f"  {t:20s}: {n}")

Speeches with at least one topic tag: 7355

Per-topic counts:
  credit              : 791
  real_estate         : 210
  monetary_policy     : 5665
  sovereign           : 133
  corporate           : 1139


In [5]:
# ── Export topic subsets ───────────────────────────────────────────────────────
OUT_DIR = r"C:\Users\Owner\OneDrive\dissertation\data\processed\topic_sentiment"
os.makedirs(OUT_DIR, exist_ok=True)

# Save full tagged file
bis.to_csv(os.path.join(OUT_DIR, "bis_topic_tagged.csv"), index=False)
print("Saved: bis_topic_tagged.csv")

# Save one file per topic (speeches with sentiment scores only)
for t in TOPICS:
    subset = bis[bis["topics"].apply(lambda x: t in x)].copy()
    subset = subset.dropna(subset=["P_pos", "P_neg", "P_neutral"])
    fname = f"bis_{t}.csv"
    subset.to_csv(os.path.join(OUT_DIR, fname), index=False)
    print(f"Saved: {fname}  ({len(subset)} speeches with scores)")

Saved: bis_topic_tagged.csv
Saved: bis_credit.csv  (548 speeches with scores)
Saved: bis_real_estate.csv  (157 speeches with scores)
Saved: bis_monetary_policy.csv  (4401 speeches with scores)
Saved: bis_sovereign.csv  (95 speeches with scores)
Saved: bis_corporate.csv  (828 speeches with scores)


In [6]:
# ── Coverage summary ───────────────────────────────────────────────────────────
summary = []
for t in TOPICS:
    subset = bis[bis["topics"].apply(lambda x: t in x)]
    scored = subset.dropna(subset=["P_pos", "P_neg", "P_neutral"])
    summary.append({
        "topic": t,
        "tagged": len(subset),
        "with_scores": len(scored),
        "pct_scored": round(100 * len(scored) / len(subset), 1) if len(subset) > 0 else 0
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
summary_df.to_csv(os.path.join(OUT_DIR, "topic_coverage_summary.csv"), index=False)
print("\nSaved: topic_coverage_summary.csv")

          topic  tagged  with_scores  pct_scored
         credit     791          548        69.3
    real_estate     210          157        74.8
monetary_policy    5665         4401        77.7
      sovereign     133           95        71.4
      corporate    1139          828        72.7

Saved: topic_coverage_summary.csv
